# Stage 9 — Official TENTACLES LOCO discovery: held out D3

Train-only official R models; save complete method artifacts for later consensus extraction, external prediction, and integrated evaluation.

## Cohort D3: training cohorts only

The **official** TENTACLES `preProcess()` and `runClassifiers()` are applied to the other three cohorts. The held-out cohort is not passed to the R fitting command. **No external prediction is produced yet**; calling this the final LOCO benchmark would be incorrect. Run one cohort at a time on Colab CPU (prefer high-RAM).

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, platform, shutil, subprocess, tarfile, tempfile, time

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    pass
ROOT=Path('/content/drive/MyDrive/CMPS355_Research')
if not (ROOT/'configs/dataset.json').is_file():
    raise FileNotFoundError('Canonical CMPS355_Research missing from mounted MyDrive')
CONFIG=json.loads((ROOT/'configs/dataset.json').read_text())
EXPECTED_DATA_SHA='8a83f20035fcac5ae5d8bfe496e4d7491e47400b61ca1080b828c8ad54e91a74'
SOURCE_SHA='069a26271e3885e015df25baa40a7a1140827d5f'
assert CONFIG['sha256'] == EXPECTED_DATA_SHA
LIB=Path('/content/cmps355_r_lib')
RSCRIPT=shutil.which('Rscript')
if RSCRIPT is None: raise RuntimeError('Rscript is not available on this Colab runtime')

def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

def installed_revision():
    env=os.environ.copy()
    env['CMPS355_R_LIB']=str(LIB)
    expr=('p<-Sys.getenv("CMPS355_R_LIB");'
          'if(dir.exists(p)).libPaths(c(p,.libPaths()));'
          'if(!requireNamespace("TENTACLES",quietly=TRUE))quit(status=2L);'
          'cat(tolower(utils::packageDescription("TENTACLES")[["RemoteSha"]]))')
    r=subprocess.run([RSCRIPT,'--vanilla','-e',expr],env=env,capture_output=True,text=True)
    return r.stdout.strip() if r.returncode==0 else None

rver=subprocess.check_output([RSCRIPT,'--vanilla','-e','cat(as.character(getRversion()))'],text=True).strip()
os_info=platform.freedesktop_os_release()
key=f"r{rver}_{os_info.get('ID','linux')}{os_info.get('VERSION_ID','unknown')}_{platform.machine()}"
cache=ROOT/'runs/environment_cache'/f'{key}.tar.gz'
manifest=ROOT/'runs/environment_cache'/f'{key}.json'
revision=installed_revision()
if revision!=SOURCE_SHA:
    if not cache.is_file() or not manifest.is_file():
        raise RuntimeError(f'Compatible pinned TENTACLES cache missing: {cache}. Run notebook 04 method verification, not a floating GitHub install.')
    meta=json.loads(manifest.read_text())
    expected={'r_version':rver,'os':os_info.get('ID','linux'),
              'os_version':os_info.get('VERSION_ID','unknown'),'arch':platform.machine()}
    if any(meta.get(k)!=v for k,v in expected.items()):
        raise RuntimeError('Cached package OS/R/architecture mismatch')
    if digest(cache)!=meta['sha256']: raise RuntimeError('Cached R library checksum mismatch')
    LIB.mkdir(parents=True,exist_ok=True)
    with tempfile.TemporaryDirectory() as tmp:
        with tarfile.open(cache,'r:gz') as tf:
            for item in tf.getmembers():
                p=Path(item.name).parts
                if not p or p[0]!='r_library' or '..' in p:
                    raise RuntimeError('Unexpected package archive member')
            tf.extractall(tmp,filter='data')
        for package in (Path(tmp)/'r_library').iterdir():
            if package.is_dir() and not (LIB/package.name).exists():
                shutil.copytree(package,LIB/package.name)
    revision=installed_revision()
if revision!=SOURCE_SHA:
    raise RuntimeError(f'Official source pin does not match: {revision}')
ENV=os.environ.copy()
ENV['CMPS355_R_LIB']=str(LIB)
ENV.update({'OPENBLAS_NUM_THREADS':'1','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1'})
print('Official TENTACLES SHA verified:',revision)
print('R version:',rver,'Cache key:',key,'Workspace:',ROOT)
try:
    import psutil
    print('Colab RAM available (GiB):',round(psutil.virtual_memory().available/1024**3,2))
except ImportError:
    pass


## Verify immutable discovery inputs and package dependencies

In [ ]:
def run_and_log(command,log_path,env=ENV):
    log_path=Path(log_path)
    with log_path.open('w',encoding='utf-8',buffering=1) as log:
        with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                              text=True,env=env,bufsize=1) as proc:
            for line in proc.stdout:
                log.write(line)
                print(line,end='',flush=True)
            return proc.wait()

def write_report(output,stage,success,extra):
    output=Path(output)
    files={}
    for p in sorted(output.rglob('*')):
        if p.is_file() and p.name!='report.json':
            files[str(p.relative_to(output))]={'sha256':digest(p),'bytes':p.stat().st_size}
    report=dict(status='PASS_CLASSIFIERS_ONLY' if success else 'FAIL_CLASSIFIERS',
                stage=stage,source_commit=SOURCE_SHA,dataset_sha256=EXPECTED_DATA_SHA,
                scope='official R fit only; no external heldout predictions or consensus yet',
                created_utc=datetime.now(timezone.utc).isoformat(),files=files,**extra)
    tmp=output/'report.json.tmp'
    tmp.write_text(json.dumps(report,indent=2,sort_keys=True)+'\n')
    os.replace(tmp,output/'report.json')
    return report
fold='d3'
folder=ROOT/'runs/tentacles_stage9_inputs'/fold
manifest_path=folder/'input_manifest.json'
if not manifest_path.is_file():
    raise FileNotFoundError('Run notebook 20 first: '+str(manifest_path))
manifest=json.loads(manifest_path.read_text())
if manifest['status']!='PASS_INPUT_PREPARATION' or manifest['heldout']!=fold:
    raise RuntimeError('Wrong or incomplete input manifest')
if manifest['dataset_sha256']!=EXPECTED_DATA_SHA:
    raise RuntimeError('Incorrect aligned dataset')
for rel, record in manifest['artifacts'].items():
    path=folder/rel
    if not path.is_file() or digest(path)!=record['sha256']:
        raise RuntimeError('Stage 9 input checksum mismatch: '+str(path))
assert set(manifest['fit_cohorts'])==({'d1','d2','d4'})
train_counts=folder/'discovery/counts.csv'
train_clinical=folder/'discovery/clinical.csv'
print('Training only:',train_counts,train_clinical)
packages=['tidymodels','workflowsets','recipes','themis','colino','Boruta','ranger',
 'nnet','kernlab','C50','bonsai','lightgbm','rpart','earth','xgboost','mixOmics','plsmod','yardstick']
expr=('p<-Sys.getenv("CMPS355_R_LIB");if(dir.exists(p)).libPaths(c(p,.libPaths()));'
      'q<-c('+','.join('"'+q+'"' for q in packages)+');'
      'missing<-q[!vapply(q,requireNamespace,logical(1),quietly=TRUE)];'
      'cat("MISSING_OPTIONAL_PACKAGES=",paste(missing,collapse=","),"\n",sep="");'
      'if(length(missing))quit(status=4L)')
pre=subprocess.run([RSCRIPT,'--vanilla','-e',expr],env=ENV,capture_output=True,text=True)
print(pre.stdout,pre.stderr)
if pre.returncode: raise RuntimeError('Missing R dependencies; no full run launched')


## Execute the official R discovery algorithm on the 3 training cohorts

In [ ]:
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
run=ROOT/'runs/tentacles_loco'/fold/stamp
run.mkdir(parents=True,exist_ok=False)
script=ROOT/'r/tentacles_stage9_loco_classifiers.R'
if not script.is_file(): raise FileNotFoundError(script)
start=time.monotonic()
rc=run_and_log([RSCRIPT,'--vanilla',str(script),str(train_counts),str(train_clinical),
                str(run),fold,SOURCE_SHA],run/'classifiers.log')
log=(run/'classifiers.log').read_text(errors='replace')
success=(rc==0 and 'LOCO_TRAIN_ONLY_PASS=TRUE' in log and
         'CLASSIFIERS_PASS=TRUE' in log and (run/'runclassifiers_finished.rds').is_file()
         and (run/'train_only_preprocessed.rds').is_file())
report=write_report(run,'LOCO_discovery_classifiers_only',success,{
  'heldout':fold,'exit_code':rc,'elapsed_wall_seconds':round(time.monotonic()-start,1),
  'input_manifest_sha256':digest(manifest_path),
  'train_counts_sha256':digest(train_counts),
  'train_meta_sha256':digest(train_clinical),
  'source_R_sha256':digest(script),
  'heldout_used_during_fit':False,
  'note':'Native official selector and fit; consensus and external predictions remain to be done'
})
print('Output:',run,'Status:',report['status'])
if not success: raise RuntimeError('TENTACLES LOCO R fit failed; inspect classifiers.log')
